# 12 — Decision-level route geometry profiles

**Goal:** Condense the 187,570 static candidate routes into 18,757 **decision-level descriptive profiles**, retaining per-match grouping, missingness and detection quality.

This is deliberately **not** an opportunity classifier or a measure of pressing suppression: a forward teammate is not necessarily a feasible route; an interior defender projection is not evidence of interception capability.

**Input:** `outputs/P2_integrated_candidate_landscape_match_{mid}.jsonl` from Notebook 11.  
**Outputs:** `outputs/P2_decision_route_profiles.jsonl` and `outputs/P2_decision_route_profiles_gate.json`.

**Important:** Quality comparisons here restrict the carrier and *teammate* to directly detected positions. Opponent tracking may still include extrapolated coordinates; the resulting subgroups must **not** be described as fully detected-only passing-lane measurements.


In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import json, math, statistics

ROOT=next((p for p in (Path.cwd().resolve(),*Path.cwd().resolve().parents) if (p/'data'/'matches').is_dir()),None)
if ROOT is None: raise FileNotFoundError('Launch notebook from the SkillCorner project.')
OUT=ROOT/'outputs'; OUT.mkdir(exist_ok=True)
files=sorted(OUT.glob('P2_integrated_candidate_landscape_match_*.jsonl'))
assert len(files)==20,f'Expected 20 integrated match files, found {len(files)}'
print('Match files:',len(files))


Match files: 20


## 1 — Parsing and invariants

The unit of description is the *individual possession-start decision*. Candidate IDs must be unique within a decision; exactly ten candidate records are expected under the previous verified scaffold. Distance and clearance remain continuous. No arbitrary "open/closed" threshold is introduced.


In [2]:
def ident(r):
    return (str(r['match_id']),int(r['period']),int(r['frame']),str(r['possession_event_id']),str(r['carrier_player_id']))
def finite(v):
    if v is None:return None
    try:
        f=float(v)
        return f if math.isfinite(f) else None
    except (TypeError,ValueError):return None

def summarize_group(rows):
    dx=[finite(r['signed_endpoint_progression_m']) for r in rows]
    distances=[finite(r['pass_segment_length_m']) for r in rows]
    assert all(x is not None for x in dx+distances)
    forward=[r for r,x in zip(rows,dx) if x>0]
    proj=[r for r in rows if r['geometry_status']=='interior_projection_present']
    no_proj=[r for r in rows if r['geometry_status']=='no_interior_projection']
    unknown=[r for r in rows if r['geometry_status'] not in ('interior_projection_present','no_interior_projection')]
    cle=[finite(r['interior_opponent_clearance_m']) for r in proj]
    assert all(x is not None for x in cle)
    detected_target=[r for r in rows if r.get('carrier_detected') is True and r.get('teammate_quality')=='detected']
    return {
        'n_candidates':len(rows),
        'n_forward_endpoints':len(forward),
        'forward_endpoint_fraction':len(forward)/len(rows),
        'n_nonforward_endpoints':len(rows)-len(forward),
        'max_signed_dx_m':max(dx),'median_signed_dx_m':statistics.median(dx),
        'median_segment_length_m':statistics.median(distances),
        'n_interior_projection':len(proj),
        'n_no_interior_projection':len(no_proj),
        'n_other_geometry_states':len(unknown),
        'n_forward_and_no_interior_projection':sum(r['geometry_status']=='no_interior_projection' for r in forward),
        'min_interior_clearance_m':min(cle) if cle else None,
        'n_carrier_target_both_detected':len(detected_target),
        'n_detected_forward_endpoints':sum(finite(r['signed_endpoint_progression_m'])>0 for r in detected_target),
        'n_detected_no_interior_projection':sum(r['geometry_status']=='no_interior_projection' for r in detected_target)
    }


## 2 — Create profiles and audit per-match consistency

The group key retains match, period, frame, event ID and carrier ID. We verify counts and quality flags but do not reduce candidate geometry to a single pseudo-probability.

All intermediate grouping is within one match, preventing accidental cross-match merges when source IDs repeat.


In [3]:
dest=OUT/'P2_decision_route_profiles.jsonl'
match_summaries=[]
aggregate=Counter()
with dest.open('w',encoding='utf-8') as out:
    for path in files:
        mid=path.name.split('_match_')[1].replace('.jsonl','')
        groups=defaultdict(list)
        with path.open(encoding='utf-8') as inp:
            for line_no,line in enumerate(inp,1):
                if not line.strip():continue
                try:r=json.loads(line)
                except json.JSONDecodeError as e:raise ValueError(f'{path.name}:{line_no}: {e}') from e
                if str(r.get('match_id'))!=mid:raise AssertionError(f'{mid}: row match ID mismatch')
                groups[ident(r)].append(r)
        seen=Counter()
        for k,rows in groups.items():
            ids=[str(r['teammate_player_id']) for r in rows]
            if len(rows)!=10 or len(set(ids))!=10:
                raise AssertionError(f'{mid} decision {k}: {len(rows)} candidates, {len(set(ids))} unique teammate IDs')
            expected_team={str(r['attacking_team_id']) for r in rows}
            if len(expected_team)!=1:raise AssertionError(f'{mid} inconsistent team within decision {k}')
            for r in rows:
                status=r['geometry_status']
                if status=='interior_projection_present' and finite(r.get('interior_opponent_clearance_m')) is None:
                    raise AssertionError(f'{mid}: missing clearance for projected defender')
                if status!='interior_projection_present' and r.get('interior_opponent_clearance_m') is not None:
                    raise AssertionError(f'{mid}: unexpected clearance for {status}')
            p={'match_id':mid,'period':k[1],'frame':k[2],'possession_event_id':k[3],
               'carrier_player_id':k[4],'attacking_team_id':next(iter(expected_team)),
               **summarize_group(rows)}
            out.write(json.dumps(p,separators=(',',':'))+'\n')
            seen['decision_profiles']+=1
            seen['candidates']+=len(rows)
            seen['forward_endpoints']+=p['n_forward_endpoints']
            seen['no_interior_projection']+=p['n_no_interior_projection']
            seen['both_detected']+=p['n_carrier_target_both_detected']
            seen['forward_and_no_interior_projection']+=p['n_forward_and_no_interior_projection']
        print(mid,dict(seen))
        match_summaries.append({'match_id':mid,**dict(seen)})
        aggregate.update(seen)
assert aggregate['decision_profiles']==18757,aggregate
assert aggregate['candidates']==187570,aggregate
assert aggregate['forward_endpoints']==103565,aggregate
assert aggregate['no_interior_projection']==4246,aggregate
assert aggregate['both_detected']==117394,aggregate
assert aggregate['forward_and_no_interior_projection']==1020,aggregate


1874553 {'decision_profiles': 956, 'candidates': 9560, 'forward_endpoints': 5407, 'no_interior_projection': 221, 'both_detected': 6158, 'forward_and_no_interior_projection': 46}
1886347 {'decision_profiles': 999, 'candidates': 9990, 'forward_endpoints': 5598, 'no_interior_projection': 256, 'both_detected': 6520, 'forward_and_no_interior_projection': 41}
1899585 {'decision_profiles': 929, 'candidates': 9290, 'forward_endpoints': 4656, 'no_interior_projection': 187, 'both_detected': 5788, 'forward_and_no_interior_projection': 37}
1925299 {'decision_profiles': 1062, 'candidates': 10620, 'forward_endpoints': 5983, 'no_interior_projection': 289, 'both_detected': 6063, 'forward_and_no_interior_projection': 77}
1927964 {'decision_profiles': 1139, 'candidates': 11390, 'forward_endpoints': 6417, 'no_interior_projection': 306, 'both_detected': 7497, 'forward_and_no_interior_projection': 61}
1953632 {'decision_profiles': 970, 'candidates': 9700, 'forward_endpoints': 6098, 'no_interior_projection'

## 3 — Report interpretation boundary

The immediate output supports exploratory descriptive statistics, **not** causal claims or match-held-out predictive evaluation. Future model units may be possession/press episodes and matches; this dataset must not be randomly split at the candidate-row level.

**Technical debt intentionally deferred:** Temporal frame quality on some matches, defender velocities, ball travel, receiver motion, observed event link semantics, pose alignment and carrying alternative construction.


In [4]:
gate={
 'created_utc':datetime.now(timezone.utc).isoformat(),
 'purpose':'Decision-level summary of verified static geometry; not passing opportunity classification',
 'n_matches':len(match_summaries),
 'totals':dict(aggregate),
 'per_match':match_summaries,
 'invariants':{
   'one_profile_per_eligible_decision':True,
   'exactly_ten_unique_teammates_per_profile':True,
   'forward_and_projection_counts_reconcile':True,
   'source_pair_counts_reconcile':True},
 'limitations':[
   'Ten candidate endpoints share one decision; no independent candidate-level statistical samples',
   'Proximity and projection are descriptive, not pressing or pass feasibility',
   'Carrier-plus-target detected subset is NOT fully detected-only lane quality; opponent coordinates may be extrapolated',
   'No arbitrary clearance or progression threshold introduced',
   'Source phases and outcomes not joined as predictors',
   'No temporal defender reach, ball speed, carrying alternatives or pose features',
   'Any downstream inference must respect match and possession/episode clustering']
}
p=OUT/'P2_decision_route_profiles_gate.json'
p.write_text(json.dumps(gate,indent=2),encoding='utf-8')
print(json.dumps({'matches':len(match_summaries),'totals':dict(aggregate)},indent=2))
print('Upload only:',p)


{
  "matches": 20,
  "totals": {
    "decision_profiles": 18757,
    "candidates": 187570,
    "forward_endpoints": 103565,
    "no_interior_projection": 4246,
    "both_detected": 117394,
    "forward_and_no_interior_projection": 1020
  }
}
Upload only: /Users/fadihaddad/Documents/GitHub/SkillCorner_opendata_soccer/outputs/P2_decision_route_profiles_gate.json
